# The Ledger on a real model: SmolLM2-135M-Instruct on IFEval

This notebook runs the experiment of *You Need Coherence* on a Colab GPU. It freezes
[SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) and compares four versions of it
on the official [IFEval](https://arxiv.org/abs/2311.07911) benchmark (541 prompts, official checkers):

| name | what it is | the question it answers |
|---|---|---|
| `base` | the frozen model, unchanged | where we start |
| `lora` | the standard way to adapt a transformer: LoRA on q and v, **the same 3.96M trainable parameters and the same data** as the Ledger | is it the architecture, or just the training? |
| `ledger` | the proposed architecture: typed obligation slots, a separate softmax read in layers 11-30, a gate on the end token | the claim |
| `ledger_joint` | the same slots and state, but read inside the model's own softmax | is the *separate* budget what matters (Fact 1)? |

Each is scored with the request alone, and with 6,000 tokens of unrelated text after it. The theory predicts that the
standard architecture loses more accuracy with the extra text than the Ledger does (prediction 3 in the README).

**How to use**

1. *Runtime → Change runtime type → T4 GPU* (or any GPU), then *Save*.
2. Run the cells in order with the ▶ buttons. Cell 1 asks for access to Google Drive: every result is saved there.
3. If Colab disconnects, reconnect, run **cell 1** again, then the cell you were on: finished steps are skipped.
4. Cell 4 downloads a zip file. Send that zip back.

Keep this browser tab open while it runs: free Colab stops sessions whose tab is closed, and any session after 12 hours.

In [ ]:
#@title 1 · GPU, code, Google Drive (about 3 minutes)
import datetime, json, os, pathlib, platform, shutil, subprocess, sys, time

def sh(cmd, env=None, cwd=None):
    """Run a shell command, streaming its output; stop on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})}, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'command failed with exit code {p.returncode}: {cmd}')

import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU: Runtime > Change runtime type > T4 GPU, Save, then run this cell again.')
props = torch.cuda.get_device_properties(0)
GPU_GB = props.total_memory / 2**30
print(f'GPU: {props.name}, {GPU_GB:.0f} GB, compute capability {props.major}.{props.minor}')

REPO = '/content/ivi-137.github.io'
CODE = f'{REPO}/research/ledger-smollm'
if not os.path.isdir(REPO):
    sh(f'git clone --depth 1 https://github.com/ivi-137/ivi-137.github.io {REPO}')
else:
    sh(f'git -C {REPO} pull --ff-only')
sh(f'{sys.executable} -m pip install -q -r requirements.txt', cwd=CODE)
sh(f'{sys.executable} setup_ifeval.py', cwd=CODE)          # official IFEval checkers and prompts, pinned commit
sh(f'{sys.executable} -m pytest -q -x tests', cwd=CODE)     # the implementation's tests, on a tiny random model

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/ledger-smollm'
except Exception as e:
    WORK = '/content/work'
    print(f'Google Drive not mounted ({e}). Saving to {WORK}, which is lost if Colab disconnects.')
os.makedirs(WORK, exist_ok=True)

# Larger GPUs take larger evaluation batches (faster, same answers); training keeps the protocol's batches.
EVAL_TOKENS = 65536 if GPU_GB < 20 else 131072 if GPU_GB < 45 else 262144
ENV = {'PYTHON': sys.executable, 'DEVICE': 'cuda', 'BATCH': '64', 'EVAL_TOKENS': str(EVAL_TOKENS),
       'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false'}
COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()

def show_summary(preset):
    from IPython.display import Markdown, display
    md = pathlib.Path(WORK, preset, 'results', 'summary.md')
    if md.exists():
        display(Markdown(f'### {preset}\n\n' + md.read_text(encoding='utf-8')))

def study(preset):
    """The whole study for one preset (see run_all.sh), resumable; results on Drive under WORK/<preset>."""
    t0 = time.time()
    tokens = EVAL_TOKENS
    while True:
        try:
            sh(f'bash run_all.sh {preset}', env={**ENV, 'EVAL_TOKENS': str(tokens), 'OUT': f'{WORK}/{preset}'}, cwd=CODE)
            break
        except RuntimeError:
            if tokens <= 8192:
                raise
            tokens //= 2  # the usual cause is GPU memory at long contexts; finished steps are kept
            print(f'\nA step failed. Retrying with evaluation batches of at most {tokens} tokens.\n')
    hours = (time.time() - t0) / 3600
    log = pathlib.Path(WORK, preset, 'timing.jsonl')
    with log.open('a', encoding='utf-8') as f:
        f.write(json.dumps({'gpu': props.name, 'hours': round(hours, 2), 'finished': datetime.datetime.now().isoformat()}) + '\n')
    print(f'\n{preset}: this session took {hours:.1f} h')
    show_summary(preset)

print(f'\nReady. Code at commit {COMMIT[:7]}; results go to {WORK}')

In [ ]:
#@title 2 · Pilot: every step once, small (about 30 to 60 minutes on a T4)
#@markdown One seed, 300 training prompts, the first 100 IFEval prompts, with 0 and 1,000 tokens of background.
#@markdown It checks the whole pipeline on this GPU and gives a first, small comparison. Its numbers are too
#@markdown few to conclude anything; they tell us the pipeline works and how fast this GPU is.
study('pilot')

In [ ]:
#@title 3 · The study (hours: see below; resumable)
#@markdown **`colab`** (recommended): the full data and training (3,000 prompts, two epochs), two seeds, `base`,
#@markdown `lora`, `ledger` and `ledger_joint`, all 541 IFEval prompts with 0 and 6,000 tokens of background.
#@markdown Roughly 6 to 10 hours on a T4 and 2 to 4 on an A100: an estimate, the pilot shows this GPU's real speed.
#@markdown If the session ends, reconnect, run cell 1, then this cell: it continues where it stopped.
#@markdown
#@markdown **`full`**: the paper's protocol, three seeds, the no-gate ablation, five context lengths.
#@markdown Several times longer: for an A100, or for many sessions.
PRESET = 'colab'  #@param ['colab', 'full']
study(PRESET)

In [ ]:
#@title 4 · Pack the results and download them (works on unfinished runs too)
import zipfile
from IPython.display import display, Markdown

presets = [p for p in ('pilot', 'colab', 'full') if pathlib.Path(WORK, p).exists()]
for p in presets:  # refresh the statistics over whatever has finished
    res = pathlib.Path(WORK, p, 'results')
    if any(res.glob('*-L*.jsonl')):
        sh(f'{sys.executable} analyze.py --results "{res}"', cwd=CODE)

import transformers
environment = {'gpu': props.name, 'gpu_gb': round(GPU_GB, 1), 'compute_capability': f'{props.major}.{props.minor}',
               'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': transformers.__version__,
               'python': platform.python_version(), 'commit': COMMIT, 'eval_tokens': EVAL_TOKENS,
               'packed': datetime.datetime.now().isoformat()}
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
zpath = f'/content/ledger-results-{stamp}.zip'
n = 0
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('environment.json', json.dumps(environment, indent=2))
    for p in presets:
        for f in sorted(pathlib.Path(WORK, p).rglob('*')):
            # results (per-prompt responses and verdicts, summaries), run logs and data statistics; not the weights
            if f.is_file() and (f.parent.name == 'results' or f.name in ('run.json', 'log.jsonl', 'stats.json', 'timing.jsonl')):
                z.write(f, str(f.relative_to(WORK)))
                n += 1
shutil.copy(zpath, WORK)  # a copy stays on Drive
print(f'{n} files, {os.path.getsize(zpath) / 2**20:.1f} MB -> {zpath} (copy in {WORK})')
for p in presets:
    show_summary(p)
from google.colab import files
files.download(zpath)

**What to send back:** the zip from cell 4 (also saved in *My Drive → ledger-smollm*). If the download does not start,
open the folder icon on the left, find the zip under `/content`, and use *⋮ → Download*.

**Reading the tables.** `prompt strict` is IFEval's headline number: the share of prompts whose every instruction is
followed. In the comparisons, `ledger vs lora` is the fair test of the architecture (same parameters, same data), and
the *length contrast* rows ask whether the Ledger loses less accuracy than a control when 6,000 tokens of unrelated
text are added (a 95% interval above zero supports prediction 3; one that includes zero does not).